# Emotion Detection from Scratch

**Author:** Suchit Mathur  
**LinkedIn:** https://www.linkedin.com/in/mathursuchit/  
**Date:** 2024/04/03  
**Dataset:** FER-2013 (Kaggle) — 35,000+ grayscale face images, 7 emotion classes

---

## Problem Statement

Facial emotion recognition has applications in mental health monitoring, customer experience analysis, and human-computer interaction. This project builds and compares three deep learning approaches:

1. **Custom CNN from scratch** — baseline model
2. **CNN with Image Augmentation** — improved generalization
3. **Transfer Learning (ResNet50)** — best performance using pre-trained weights

Target: classify faces into 7 emotions — Angry, Disgust, Fear, Happy, Neutral, Sad, Surprise

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'tensorflow', 'numpy', 'matplotlib', 'seaborn', 'opencv-python', 'Pillow', 'scikit-learn'],
    stdout=subprocess.DEVNULL)

0

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow version: 2.21.0
GPU available: False


## 1. Load & Explore Data

In [ ]:
TRAIN_DIR = 'data/train'
TEST_DIR  = 'data/test'
EMOTIONS  = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
IMG_SIZE  = 224
BATCH     = 32

# Count images per class
print('Training set:')
train_counts = {}
for emotion in EMOTIONS:
    count = len(os.listdir(os.path.join(TRAIN_DIR, emotion)))
    train_counts[emotion] = count
    print(f'  {emotion:10s}: {count:,} images')

print(f'\nTotal train images: {sum(train_counts.values()):,}')

print('\nTest set:')
for emotion in EMOTIONS:
    count = len(os.listdir(os.path.join(TEST_DIR, emotion)))
    print(f'  {emotion:10s}: {count:,} images')

In [ ]:
# Class distribution
plt.figure(figsize=(10, 4))
plt.bar(train_counts.keys(), train_counts.values(), color='steelblue')
plt.title('Training Images per Emotion Class')
plt.xlabel('Emotion')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample image from each class
fig, axes = plt.subplots(1, 7, figsize=(16, 3))
for i, emotion in enumerate(EMOTIONS):
    img_path = os.path.join(TRAIN_DIR, emotion,
                            os.listdir(os.path.join(TRAIN_DIR, emotion))[0])
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(emotion.capitalize())
    axes[i].axis('off')
plt.suptitle('Sample Image per Emotion', fontsize=13)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Model 1 — Custom CNN from Scratch

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH,
    class_mode='categorical', shuffle=True
)
test_gen = datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH,
    class_mode='categorical', shuffle=False
)

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 1), num_classes=7):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_model = build_custom_cnn()
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

history_cnn = cnn_model.fit(
    train_gen, epochs=30,
    validation_data=test_gen,
    callbacks=callbacks
)

## 3. Model 2 — CNN with Image Augmentation

In [ ]:
aug_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

aug_train_gen = aug_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale', batch_size=BATCH,
    class_mode='categorical', shuffle=True
)

cnn_aug_model = build_custom_cnn()
cnn_aug_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history_aug = cnn_aug_model.fit(
    aug_train_gen, epochs=30,
    validation_data=test_gen,
    callbacks=callbacks
)

## 4. Model 3 — Transfer Learning (ResNet50)

In [ ]:
# ResNet50 expects RGB input
rgb_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1
)

rgb_train_gen = rgb_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH,
    class_mode='categorical', shuffle=True
)
rgb_test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH,
    class_mode='categorical', shuffle=False
)

In [ ]:
base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False  # freeze base

resnet_model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

callbacks_resnet = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ModelCheckpoint('Final_Resnet50_Best_model.keras', save_best_only=True, verbose=1)
]

history_resnet = resnet_model.fit(
    rgb_train_gen, epochs=30,
    validation_data=rgb_test_gen,
    callbacks=callbacks_resnet
)

## 5. Results & Comparison

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(f'{title} — Accuracy')
    axes[0].legend()
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(f'{title} — Loss')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_cnn, 'Custom CNN')
plot_history(history_aug, 'CNN + Augmentation')
plot_history(history_resnet, 'ResNet50 Transfer Learning')

In [ ]:
import pandas as pd

results = {
    'Custom CNN':           max(history_cnn.history['val_accuracy']),
    'CNN + Augmentation':   max(history_aug.history['val_accuracy']),
    'ResNet50 (Transfer)':  max(history_resnet.history['val_accuracy'])
}

results_df = pd.DataFrame(results.items(), columns=['Model', 'Val Accuracy'])
results_df['Val Accuracy'] = results_df['Val Accuracy'].map('{:.2%}'.format)
print(results_df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.bar(results.keys(), results.values(), color=['#4C72B0', '#55A868', '#C44E52'])
plt.title('Model Comparison — Validation Accuracy')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix for best model (ResNet50)
best_model = tf.keras.models.load_model('Final_Resnet50_Best_model.keras')
y_pred = np.argmax(best_model.predict(rgb_test_gen), axis=1)
y_true = rgb_test_gen.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[e.capitalize() for e in EMOTIONS],
            yticklabels=[e.capitalize() for e in EMOTIONS])
plt.title('Confusion Matrix — ResNet50')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=[e.capitalize() for e in EMOTIONS]))

## 6. Conclusions

- **ResNet50 transfer learning outperformed** both custom CNN variants — pre-trained ImageNet weights give the model a strong foundation for recognizing visual patterns in faces
- **Image augmentation** improved generalization over the baseline CNN by reducing overfitting
- **Class imbalance** (disgust has far fewer samples) affects per-class performance — future work could use class weights or oversampling
- **Deployment:** The best model is served via a Streamlit app with real-time webcam and image upload support